# Complete Pydantic Guide: From Basics to Advanced

## A Comprehensive Tutorial on Pydantic Data Validation Library

**Version:** 1.0  
**Based on:** hiteshchoudhary/pydantic-Hindi Repository  
**Last Updated:** 2026-05-01

---

## Table of Contents
1. [Introduction](#introduction)
2. [Foundation - Basic Models](#foundation)
3. [Fields and Validation](#fields-validation)
4. [Model Behavior - Validators](#validators)
5. [Nested Models](#nested-models)
6. [Serialization](#serialization)
7. [FastAPI Integration](#fastapi)
8. [Best Practices](#best-practices)

---

# 1. Introduction to Pydantic

## What is Pydantic?

Pydantic is a Python library that uses Python type hints to validate data. It enforces type hints at runtime and provides helpful error messages when data validation fails.

### Key Features:
- **Type Validation**: Enforces type hints at runtime
- **Data Serialization**: Converts models to dict/JSON
- **Error Handling**: Provides detailed validation errors
- **Performance**: Fast data validation
- **Extensible**: Custom validators and computed fields

### Installation

```bash
pip install pydantic
# For FastAPI integration
pip install pydantic fastapi uvicorn
# For email validation
pip install pydantic[email]
```

In [ ]:
# Import required libraries
from pydantic import BaseModel, Field, field_validator, model_validator, computed_field, ConfigDict
from typing import List, Dict, Optional
from datetime import datetime

print("✅ Pydantic library imported successfully!")

---

# 2. Foundation - Basic Pydantic Models

## What is a Pydantic Model?

A Pydantic model is a Python class that inherits from `BaseModel`. It defines the structure of your data with type hints.

### Basic Example

In [ ]:
# Example 1: Simple User Model
class User(BaseModel):
    id: int
    name: str
    is_active: bool

# Creating an instance from dictionary
input_data = {'id': 101, 'name': "ChaiCode", 'is_active': True}
user = User(**input_data)

print("User Model:")
print(f"  ID: {user.id}")
print(f"  Name: {user.name}")
print(f"  Active: {user.is_active}")
print(f"\nFull Output:")
print(user)

### Type Validation in Action

In [ ]:
# Pydantic automatically converts compatible types
user2 = User(id="102", name="Alice", is_active="yes")
print("\nType Conversion:")
print(f"ID type: {type(user2.id).__name__} = {user2.id}")
print(f"Is Active type: {type(user2.is_active).__name__} = {user2.is_active}")

In [ ]:
# What happens with invalid data?
try:
    invalid_user = User(id="not_a_number", name="Bob", is_active=True)
except Exception as e:
    print("❌ Validation Error:")
    print(str(e))

### Product Model Example

In [ ]:
# Product Model - Simple but effective
class Product(BaseModel):
    id: int
    name: str
    price: float
    in_stock: bool = True  # Default value

# Create product instances
product1 = Product(id=1, name="Laptop", price=999.99)
product2 = Product(id=2, name="Mouse", price=29.99, in_stock=False)

print("Product 1:")
print(product1)
print("\nProduct 2:")
print(product2)

---

# 3. Fields and Validation

## Using Field() for Advanced Constraints

The `Field()` function allows you to add constraints, descriptions, and examples to model fields.

### String Field Constraints

In [ ]:
# Employee Model with Field Constraints
class Employee(BaseModel):
    id: int
    name: str = Field(
        ...,  # Required field
        min_length=3,
        max_length=50,
        description="Employee Name",
        example="Hitesh Choudhary"
    )
    department: Optional[str] = Field(
        default='General',
        description="Department of employee"
    )
    salary: float = Field(
        ...,
        ge=10000,  # Greater than or equal
        description="Annual salary (minimum 10000)"
    )

# Valid employee
emp1 = Employee(id=1, name="Alice", salary=50000)
print("✅ Valid Employee:")
print(emp1)
print(f"\nDepartment: {emp1.department} (default value)")

In [ ]:
# Invalid - name too short
try:
    emp_invalid = Employee(id=2, name="Al", salary=50000)
except Exception as e:
    print("❌ Validation Error (name too short):")
    print(str(e))

In [ ]:
# Invalid - salary too low
try:
    emp_low_salary = Employee(id=3, name="Bob", salary=5000)
except Exception as e:
    print("❌ Validation Error (salary too low):")
    print(str(e))

### Complex Field Types

In [ ]:
# Model with List and Dict types
class Cart(BaseModel):
    user_id: int
    items: List[str]  # List of item names
    quantities: Dict[str, int]  # Dict mapping item name to quantity

class BlogPost(BaseModel):
    title: str
    content: str
    image_url: Optional[str] = None  # Optional field

# Creating instances
cart = Cart(
    user_id=101,
    items=["Laptop", "Mouse", "Keyboard"],
    quantities={"Laptop": 1, "Mouse": 2, "Keyboard": 1}
)

blog = BlogPost(
    title="Pydantic Guide",
    content="Learn Pydantic from scratch!"
    # image_url is optional, can be omitted
)

print("Cart:")
print(cart)
print("\nBlog Post:")
print(blog)

### Field Constraints Cheatsheet

| Constraint | Use Case | Example |
|-----------|----------|----------|
| `min_length` | Minimum string length | `Field(..., min_length=3)` |
| `max_length` | Maximum string length | `Field(..., max_length=50)` |
| `ge` | Greater than or equal | `Field(..., ge=0)` |
| `le` | Less than or equal | `Field(..., le=100)` |
| `gt` | Greater than | `Field(..., gt=0)` |
| `lt` | Less than | `Field(..., lt=100)` |
| `pattern` | Regex pattern | `Field(..., pattern=r'^[A-Z]')` |
| `default` | Default value | `Field(default='N/A')` |
| `...` | Required field | `Field(...)` |
| `description` | Field description | `Field(..., description='User name')` |
| `example` | Example value | `Field(..., example='John')` |

---

# 4. Model Behavior - Validators and Computed Fields

## Field Validators

Use `@field_validator` to add custom validation logic to specific fields.

In [ ]:
# User model with field validator
class User(BaseModel):
    username: str

    @field_validator('username')
    def username_length(cls, v):
        if len(v) < 4:
            raise ValueError("Username must be at least 4 characters")
        return v

# Valid username
user1 = User(username="alice123")
print("✅ Valid Username:")
print(user1)

# Invalid username
try:
    user2 = User(username="bob")
except Exception as e:
    print("\n❌ Invalid Username:")
    print(str(e))

## Model Validators

Use `@model_validator` to validate multiple fields together.

In [ ]:
# Signup with password matching validator
class SignupData(BaseModel):
    password: str
    confirm_password: str

    @model_validator(mode='after')
    def password_match(cls, values):
        if values.password != values.confirm_password:
            raise ValueError('Passwords do not match')
        return values

# Valid signup
signup1 = SignupData(password="secure123", confirm_password="secure123")
print("✅ Valid Signup:")
print(f"Password matches: {signup1.password == signup1.confirm_password}")

# Invalid signup
try:
    signup2 = SignupData(password="secure123", confirm_password="different")
except Exception as e:
    print("\n❌ Invalid Signup:")
    print(str(e))

## Computed Fields

Use `@computed_field` to create fields that are calculated from other fields.

In [ ]:
# Product with computed total price
class Product(BaseModel):
    price: float
    quantity: int

    @computed_field
    @property
    def total_price(self) -> float:
        """Calculate total price = price × quantity"""
        return self.price * self.quantity

product = Product(price=100.0, quantity=5)
print("Product with Computed Field:")
print(f"Price: ${product.price}")
print(f"Quantity: {product.quantity}")
print(f"Total Price: ${product.total_price}")
print(f"\nFull model:")
print(product)

---

# 5. Nested Models

## Simple Nested Models

Create hierarchical data structures by nesting models.

In [ ]:
# Address Model
class Address(BaseModel):
    street: str
    city: str
    postal_code: str

# User Model with nested Address
class User(BaseModel):
    id: int
    name: str
    address: Address  # Nested model

# Creating an address
address = Address(
    street="123 Main Street",
    city="Jaipur",
    postal_code="10001"
)

# Creating a user with the address
user = User(
    id=1,
    name="Hitesh",
    address=address
)

print("User with Nested Address:")
print(user)
print(f"\nAccessing nested fields:")
print(f"City: {user.address.city}")
print(f"Postal Code: {user.address.postal_code}")

## Self-Referencing Models (Recursive)

Create models that reference themselves.

In [ ]:
# Comment model with optional replies (self-reference)
class Comment(BaseModel):
    id: int
    content: str
    replies: Optional[List['Comment']] = None  # Forward reference

# Rebuild the model to resolve forward references
Comment.model_rebuild()

# Create nested comments
comment = Comment(
    id=1,
    content="First Comment",
    replies=[
        Comment(id=2, content="Reply 1"),
        Comment(id=3, content="Reply 2", replies=[
            Comment(id=4, content="Nested Reply")
        ])
    ]
)

print("Comment with Nested Replies:")
print(comment)
print(f"\nMain comment has {len(comment.replies)} replies")
print(f"Second reply has {len(comment.replies[1].replies)} nested replies")

---

# 6. Serialization

## Converting Models to Python Objects and JSON

Pydantic provides methods to serialize models into different formats.

In [ ]:
# Complex model for serialization examples
class Address(BaseModel):
    street: str
    city: str
    zip_code: str

class User(BaseModel):
    id: int
    name: str
    email: str
    is_active: bool = True
    createdAt: datetime
    address: Address
    tags: List[str] = []

    model_config = ConfigDict(
        json_encoders={datetime: lambda v: v.strftime('%d-%m-%Y %H:%M:%S')}
    )

# Create a user instance
user = User(
    id=1,
    name="Hitesh",
    email="hitesh@hc.com",
    createdAt=datetime(2024, 3, 15, 14, 30),
    address=Address(
        street="123 Tech Street",
        city="Jaipur",
        zip_code="001144"
    ),
    is_active=True,
    tags=["premium", "subscriber"]
)

print("User Model Instance:")
print(user)

### model_dump() - Convert to Dictionary

In [ ]:
# Convert model to Python dictionary
python_dict = user.model_dump()

print("\n📦 model_dump() - Python Dictionary:")
print("="*50)
print(python_dict)
print(f"\nType: {type(python_dict)}")

### model_dump_json() - Convert to JSON String

In [ ]:
# Convert model to JSON string
json_str = user.model_dump_json()

print("\n📄 model_dump_json() - JSON String:")
print("="*50)
print(json_str)
print(f"\nType: {type(json_str)}")

### Serialization Options

In [ ]:
# Exclude specific fields
print("\n🚫 Exclude fields:")
print(user.model_dump(exclude={'email', 'tags'}))

# Include only specific fields
print("\n✅ Include only specific fields:")
print(user.model_dump(include={'id', 'name', 'email'}))

# Pretty print JSON
print("\n🎨 Pretty JSON:")
import json
print(json.dumps(user.model_dump(), indent=2, default=str))

---

# 7. FastAPI Integration

## Using Pydantic Models with FastAPI

Pydantic models are perfect for FastAPI request/response validation.

In [ ]:
# Example FastAPI models (code only - for reference)
example_code = '''
from fastapi import FastAPI, Depends
from pydantic import BaseModel, EmailStr

app = FastAPI()

# Request Model
class UserSignup(BaseModel):
    username: str
    email: EmailStr  # Validates email format
    password: str

# Settings Model
class Settings(BaseModel):
    app_name: str = "Chai App"
    admin_email: str = 'admin@chai.com'

def get_settings():
    return Settings()

# POST endpoint with request validation
@app.post('/signup')
def signup(user: UserSignup):  # Pydantic validates input
    return {'message': f'User {user.username} signed up successfully'}

# GET endpoint with dependency injection
@app.get('/settings')
def get_settings_endpoint(settings: Settings = Depends(get_settings)):
    return settings

# Run with: uvicorn main:app --reload
'''

print(example_code)

## FastAPI + Pydantic Benefits

| Feature | Benefit |
|---------|----------|
| **Automatic Validation** | FastAPI validates request data using Pydantic |
| **Error Handling** | Automatic 422 errors for validation failures |
| **Documentation** | OpenAPI docs auto-generated from models |
| **Type Hints** | IDE autocomplete and type checking |
| **Response Models** | Auto-serialize response to JSON |
| **Dependencies** | Use Depends() for Pydantic models |

---

# 8. Best Practices and Advanced Tips

## Best Practices for Pydantic

In [ ]:
best_practices = {
    "1. Use Type Hints": "Always specify types for fields",
    "2. Use Field() for Constraints": "Add validation with Field(min_length, ge, etc.)",
    "3. Use Optional for Nullable Fields": "Optional[str] for fields that can be None",
    "4. Create Base Models": "Use inheritance for shared fields",
    "5. Use Validators": "Add custom validation with @field_validator",
    "6. Use ConfigDict": "Configure model behavior with model_config",
    "7. Serialize Properly": "Use model_dump() and model_dump_json()",
    "8. Handle Errors": "Catch ValidationError exceptions",
    "9. Document Models": "Use Field(description=...) for API docs",
    "10. Use Computed Fields": "Calculate derived values with @computed_field"
}

print("\n🎯 PYDANTIC BEST PRACTICES")
print("="*60)
for i, (practice, benefit) in enumerate(best_practices.items(), 1):
    print(f"\n{practice}")
    print(f"   → {benefit}")

## Model Inheritance Example

In [ ]:
# Base model with common fields
class BaseEntity(BaseModel):
    id: int
    created_at: datetime
    updated_at: datetime

# Inherit from base model
class BlogPost(BaseEntity):
    title: str
    content: str
    author: str

class Comment(BaseEntity):
    text: str
    post_id: int
    author: str

# Using inherited models
post = BlogPost(
    id=1,
    created_at=datetime.now(),
    updated_at=datetime.now(),
    title="Pydantic Guide",
    content="Learn Pydantic...",
    author="Hitesh"
)

print("Model with Inheritance:")
print(post)

## Common Validation Patterns

In [ ]:
import re

# Email validation
class ContactInfo(BaseModel):
    email: str = Field(..., pattern=r'^[\w\.-]+@[\w\.-]+\.\w+$')
    phone: str = Field(..., pattern=r'^\d{10}$')
    website: Optional[str] = Field(None, pattern=r'^https?://')

# Valid contact info
contact = ContactInfo(
    email="user@example.com",
    phone="9876543210",
    website="https://example.com"
)

print("Valid Contact Info:")
print(contact)

# Invalid email
try:
    invalid = ContactInfo(
        email="invalid.email",
        phone="9876543210"
    )
except Exception as e:
    print("\n❌ Invalid email pattern")

## Configuration Options

In [ ]:
# Advanced model configuration
class StrictModel(BaseModel):
    name: str
    age: int
    
    model_config = ConfigDict(
        # Validate on assignment
        validate_assignment=True,
        # Use field names from model (not title-cased)
        use_attribute_docstrings=True,
        # Allow population by field name
        populate_by_name=True
    )

model = StrictModel(name="Alice", age=30)
print("Configured Model:")
print(model)

# Validate on assignment
try:
    model.age = "not a number"  # Will raise validation error
except Exception as e:
    print("\n✅ Validation on assignment caught the error")

---

# Summary and Key Takeaways

## What You've Learned

### ✅ Foundation
- Create Pydantic models with `BaseModel`
- Define fields with type hints
- Automatic type validation and coercion

### ✅ Fields & Validation
- Use `Field()` for constraints and descriptions
- String constraints: `min_length`, `max_length`, `pattern`
- Numeric constraints: `ge`, `le`, `gt`, `lt`
- Complex types: `List`, `Dict`, `Optional`

### ✅ Model Behavior
- `@field_validator` for single field validation
- `@model_validator` for multi-field validation
- `@computed_field` for calculated fields

### ✅ Nested Models
- Nest models within models
- Self-referencing models with forward references
- Build complex data hierarchies

### ✅ Serialization
- `model_dump()` → Python dictionary
- `model_dump_json()` → JSON string
- Exclude/include specific fields

### ✅ FastAPI Integration
- Use models for request validation
- Auto-generate OpenAPI documentation
- Handle errors automatically

---

## Common Patterns to Remember

```python
# Basic model
class Model(BaseModel):
    field: str

# With constraints
class Model(BaseModel):
    field: str = Field(..., min_length=1, max_length=100)

# With validation
class Model(BaseModel):
    field: str
    
    @field_validator('field')
    def validate_field(cls, v):
        return v

# Serialization
obj = Model(field="value")
dict_data = obj.model_dump()
json_str = obj.model_dump_json()
```

---

## Next Steps

1. **Build a project**: Create a real FastAPI application
2. **Explore plugins**: Use pydantic-extra-types for more validators
3. **Read documentation**: Check pydantic.dev for v2 features
4. **Practice**: Create models for your use cases
5. **Contribute**: Help improve Pydantic on GitHub

---

## Resources

- 📚 **Official Docs**: https://docs.pydantic.dev
- 🎬 **Video Course**: hiteshchoudhary/pydantic-Hindi
- 💻 **GitHub**: https://github.com/pydantic/pydantic
- 🚀 **FastAPI**: https://fastapi.tiangolo.com

---

## Happy Coding! 🎉

Pydantic makes data validation in Python simple, elegant, and powerful. Use it in your next project!
